In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
df = pd.read_csv("diabetics.csv")

# Select numerical features
features = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]

# Remove rows with missing values in the required columns
df = df.dropna(subset=features + ["Outcome"])

# A. UNIVARIATE ANALYSIS
print("Outcome frequency:")
print(df["Outcome"].value_counts().sort_index())

stats = pd.DataFrame({
    "Mean": df[features].mean(),
    "Median": df[features].median(),
    "Mode": df[features].mode().iloc[0],
    "Variance": df[features].var(),
    "Standard Deviation": df[features].std(),
    "Skewness": df[features].apply(skew),
    "Kurtosis": df[features].apply(kurtosis)
})

print("\nUnivariate Statistics:")
print(stats.round(3))

print("\nFrequency of Glucose values:")
print(df["Glucose"].value_counts().head(10))

# B. BIVARIATE LINEAR REGRESSION
X = df[["Glucose"]]
y = df["BMI"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear = LinearRegression()
linear.fit(X_train, y_train)
pred = linear.predict(X_test)

print("\nBivariate Linear Regression")
print("BMI =", round(linear.intercept_, 3), "+",
      round(linear.coef_[0], 3), "* Glucose")
print("R2:", round(r2_score(y_test, pred), 3))
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 3))

plt.scatter(X_test["Glucose"], y_test, label="Actual")
plt.scatter(X_test["Glucose"], pred, label="Predicted")
plt.xlabel("Glucose")
plt.ylabel("BMI")
plt.title("Linear Regression: Glucose vs BMI")
plt.legend()
plt.show()

# B. BIVARIATE LOGISTIC REGRESSION
X = df[["Glucose"]]
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logistic = make_pipeline(StandardScaler(), LogisticRegression())
logistic.fit(X_train, y_train)
pred = logistic.predict(X_test)

print("\nBivariate Logistic Regression")
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

# C. MULTIPLE LINEAR REGRESSION
multiple_features = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "DiabetesPedigreeFunction", "Age"
]

X = df[multiple_features]
y = df["BMI"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

multiple_linear = LinearRegression()
multiple_linear.fit(X_train, y_train)
pred = multiple_linear.predict(X_test)

print("\nMultiple Linear Regression")
print("R2:", round(r2_score(y_test, pred), 3))
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 3))

# D. MULTIPLE LOGISTIC REGRESSION
X = df[multiple_features]
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

multiple_logistic = make_pipeline(StandardScaler(), LogisticRegression())
multiple_logistic.fit(X_train, y_train)
pred = multiple_logistic.predict(X_test)

print("\nMultiple Logistic Regression")
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))